## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Nelder Mead
- Output

In [2]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [11]:
from pathlib import Path

In [12]:
current_dir = Path.cwd()

In [15]:
competition_root = current_dir.parents[1]

In [18]:
# competition_root

In [19]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [20]:
train = pd.read_csv(competition_path/'data/train.csv')
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/'data/test.csv')
test_data = test[["id"]]

submit = pd.read_csv(competition_path/'data/sample_submission.csv')

In [22]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [23]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [28]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [30]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [32]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [33]:
'''
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])
'''

'\nX = X.drop(columns=["log_5cv"])\nX_test = X_test.drop(columns=["log_5cv"])\n'

In [34]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [35]:
num_models

36

In [36]:
# 2. Define the objective function with a penalty constraint
def objective(weights, predictions, y_true):
    # Standardize weights so they sum to 1 for the actual blend calculation
    # This helps Nelder-Mead evaluate valid combinations even during search
    sum_w = np.sum(weights)
    if sum_w == 0:
        normalized_weights = weights
    else:
        normalized_weights = weights / sum_w
    
    # Calculate the blended prediction
    blend = np.tensordot(normalized_weights, predictions, axes=((0), (1)))
    
    # Calculate our metric (Nelder-Mead MINIMIZES, so we do 1 - AUC)
    score = 1.0 - roc_auc_score(y_true, blend)
    
    # Penalty: If weights go negative, add a heavy penalty to steer the optimizer away
    penalty = 0
    if np.any(weights < 0):
        penalty += 10.0 * np.abs(np.min(weights))

    l1_lambda = 0.001
    penalty += l1_lambda * np.sum(np.abs(normalized_weights))
        
    return score + penalty

In [37]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [38]:
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)

In [39]:
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)

In [40]:
best_weights

array([0.02793002, 0.03808256, 0.00093521, 0.00906571, 0.00040212,
       0.04880211, 0.02579554, 0.03041818, 0.02427823, 0.03906704,
       0.02831104, 0.03278759, 0.01180998, 0.02821122, 0.00027512,
       0.03171713, 0.00200301, 0.01280215, 0.02991994, 0.03248834,
       0.00045117, 0.02061483, 0.06279733, 0.02969319, 0.04032371,
       0.0325347 , 0.02799625, 0.03995426, 0.05293553, 0.0306236 ,
       0.03386907, 0.05413595, 0.02695234, 0.03111241, 0.0368546 ,
       0.02404883])

In [41]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")

Optimization Successful: False
Optimized Blending Weights: [0.0279 0.0381 0.0009 0.0091 0.0004 0.0488 0.0258 0.0304 0.0243 0.0391
 0.0283 0.0328 0.0118 0.0282 0.0003 0.0317 0.002  0.0128 0.0299 0.0325
 0.0005 0.0206 0.0628 0.0297 0.0403 0.0325 0.028  0.04   0.0529 0.0306
 0.0339 0.0541 0.027  0.0311 0.0369 0.024 ]
Best ROC-AUC Score: 0.9454


In [42]:
# Compute the final blended predictions
final_blend = np.dot(X_test.values, best_weights)

In [43]:
submit["is_fraud"] = final_blend

In [44]:
submit.to_csv(competition_path/"Blending Nelder Mead v6.csv",index=False)

In [45]:
# Drop high correlated variable

In [46]:
'''
# 1. Calculate the correlation matrix (if not already done)
corr_matrix = X.corr().abs()

# 2. Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
'''

'\n# 1. Calculate the correlation matrix (if not already done)\ncorr_matrix = X.corr().abs()\n\n# 2. Select the upper triangle of the correlation matrix\nupper_tri = corr_matrix.where(\n    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)\n)\n'

In [47]:
'''
# 3. Find features with correlation greater than the threshold (e.g., 0.95)
threshold = 0.97
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
'''

'\n# 3. Find features with correlation greater than the threshold (e.g., 0.95)\nthreshold = 0.97\nto_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]\n'

In [48]:
'''
# 4. Drop the features from your training and test sets
X_reduced = X.drop(columns=to_drop)
X_test_reduced = X_test.drop(columns=to_drop)
'''

'\n# 4. Drop the features from your training and test sets\nX_reduced = X.drop(columns=to_drop)\nX_test_reduced = X_test.drop(columns=to_drop)\n'

In [49]:
'''
oof_matrix = X_reduced.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]
'''

'\noof_matrix = X_reduced.values \ny_true_array = y.values\nnum_models = oof_matrix.shape[1]\n'

In [50]:
'''
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)
'''

'\n# 3. Setup and Run the Optimizer\n# Start with equal weights for all 3 models\ninit_weights = np.array([1/num_models]*num_models)\n'

In [51]:
'''
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X_reduced, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)
'''

"\n# Run Nelder-Mead\nres = minimize(\n    objective, \n    init_weights, \n    args=(X_reduced, y), \n    method='Nelder-Mead',\n    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}\n)\n"

In [52]:
'''
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)
'''

'\n# 4. Process and Normalize the Final Best Weights\nbest_raw_weights = res.x\nbest_weights = best_raw_weights / np.sum(best_raw_weights)\n'

In [53]:
'''
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")
'''

'\n# 5. Print Results\nprint("Optimization Successful:", res.success)\nprint(f"Optimized Blending Weights: {np.round(best_weights, 4)}")\nprint(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")\n'

In [54]:
'''
# Compute the final blended predictions
final_blend = np.dot(X_test_reduced.values, best_weights)
'''

'\n# Compute the final blended predictions\nfinal_blend = np.dot(X_test_reduced.values, best_weights)\n'

In [55]:
'''
submit["is_fraud"] = final_blend
submit.to_csv("Blending Nelder Mead v5.csv",index=False)
'''

'\nsubmit["is_fraud"] = final_blend\nsubmit.to_csv("Blending Nelder Mead v5.csv",index=False)\n'